In [50]:
import json
import os
import time
from datetime import datetime, timezone, timedelta

import numpy as np
import pandas as pd

import torch

In [49]:
events = pd.read_parquet("events.parquet")
events = events.rename(columns = {
    "magnitude": "m",
    "timestamp": "t"
})
events.head()

,t,m
0,1460248219,4.5
1,1460254474,5.7
2,1460262120,4.9
3,1460270451,4.7
4,1460272281,5.5


### Self-Exciting Marked Point Process

We model the intensity as
$$\lambda(t \,|\, t_{i} < t) = \mu + A \sum_{t_{i} < t} e^{\gamma(m_{i} - m_{0})}(p-1)c^{p-1}(t - t_{i} + c)^{-p}$$
and estimate the model parameters by minimizing the likelihood function:
$$\ell(\theta)=\sum_{j=1}^{n} \log \lambda(t_j)-\int_0^T \lambda(s)\,ds$$
which is given by
$$\ell(\mu, A, \gamma, p, m_{0}, T) = \sum_{j=1}^{n} \log \left[ \mu + A (p-1)c^{p-1}  \sum_{i:t_i<t_j} e^{\gamma(m_i-m_0)} (t_j-t_i+c)^{-p} \right] - \mu T - A \sum_{i=1}^{n} e^{\gamma(m_i-m_0)} \left[1-\left(\frac{c}{T-t_i+c}\right)^{p-1}\right].$$
We introduce a window variable to make the optimization lighter:
$$\ell(\mu, A, \gamma, p, m_{0}, T, W) = \sum_{j=1}^{n} \log \left[ \mu + A (p-1)c^{p-1} \sum_{i:0<t_{j}-t_{i}<W} e^{\gamma(m_i-m_0)} (t_j-t_i+c)^{-p} \right] - \mu T - A \sum_{i=1}^{n} e^{\gamma(m_i-m_0)} \left[1-\left(\frac{c}{T-t_i+c}\right)^{p-1}\right].$$

In [ ]:
def pair_term(ti: float, tj: float, mi: float, m0: float, gamma: float, c: float, p: float) -> float:
    return torch.exp(gamma * (mi - m0)) * (tj - ti + c) ** (-p)
                   
def sums_term(tj: float, mi: float, m0: float, gamma: float, c: float, p: float, w: float) -> float:

    return torch.exp(gamma * (mi - m0)) * (1 - (c / (T - tj + c)) ** (p - 1))
    

In [ ]:
# subset events such that t is between tj